# Grammar Scoring: exploratory data analysis
## 1. Title and objective
Predicting spoken English grammar scores is a regression task. This notebook
examines the training target range, table integrity, and WAV header metadata
before modeling. RMSE and Pearson correlation are the competition metrics;
no modeling results are reported here. Test labels are placeholders, never targets.

## 2. Imports and configuration
Run from the repository root or `notebooks/` using the project environment.
The configured data directory can be overridden with `GRAMMAR_DATA_DIR` before
starting Python. Only WAV headers are inspected; audio samples are not processed.
The seed records deterministic configuration; no random jitter is needed.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import pearsonr, spearmanr

project_root = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src" / "grammar_scoring").is_dir()
)
sys.path.insert(0, str(project_root / "src"))

from grammar_scoring.data.dataset import (  # noqa: E402
    DATASET_DIR,
    TEST_AUDIO_DIR,
    load_sample_submission,
    load_test_dataframe,
    load_train_dataframe,
)
from grammar_scoring.data.inspection import inspect_dataset_audio  # noqa: E402

SEED = 42
SHORT_SECONDS = 30
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True})
print(f"Data directory: {DATASET_DIR}; seed: {SEED}")

## 3. Load dataset
Inspect all three tables as supplied. The test `label` column is retained only
for documenting placeholders and excluded from all target analyses.

In [ ]:
train = load_train_dataframe()
test = load_test_dataframe()
submission = load_sample_submission()
tables = {"train": train, "test": test, "sample_submission": submission}
for name, frame in tables.items():
    print(f"{name}: shape={frame.shape}, columns={frame.columns.tolist()}")
    display(frame.dtypes.rename("dtype").to_frame())
    display(frame.head())
print("Test placeholder values:", test["label"].unique().tolist())
assert test["label"].eq(-1).all(), "Reinvestigate unexpected test placeholders"
print("Every test label is -1: a placeholder, never ground truth.")

## 4. Dataset integrity summary
Filename identity is split-specific: `train/audio_0.wav` and
`test/audio_0.wav` represent different samples. Overlaps below are basename
comparisons for packaging inspection, not evidence of shared sample identity.
No cross-split sample join is performed. Submission alignment is a packaging
issue requiring investigation before final submission; this notebook does not
repair it. “Readable” refers to the existing WAV header reader, not full decoding.

In [ ]:
inspections = {item.split: item for item in inspect_dataset_audio()}
metadata = {split: item.metadata for split, item in inspections.items()}
train_metadata = metadata["train"]
test_metadata = metadata["test"]
audio_metadata = pd.concat(metadata.values(), ignore_index=True)
assert not audio_metadata.duplicated(["split", "filename"]).any()

integrity = {}
for split, item in inspections.items():
    frame = item.metadata
    integrity[split.title()] = {
        "CSV rows": item.csv_rows,
        "WAV files": len(frame) + len(item.errors),
        "Readable WAVs": len(frame),
        "Sample rate (Hz)": sorted(frame["sample_rate"].unique().tolist()),
        "Channels": sorted(frame["num_channels"].unique().tolist()),
        "Sample width (bytes)": sorted(frame["sample_width_bytes"].unique().tolist()),
    }
    if item.errors:
        display(pd.DataFrame(item.errors, columns=["path", "error"]))
display(pd.DataFrame(integrity))

names = {name: set(frame["filename"]) for name, frame in tables.items()}
test_wav_names = {
    path.name
    for path in TEST_AUDIO_DIR.iterdir()
    if path.is_file() and path.suffix.lower() == ".wav"
}
missing_submission_audio = names["sample_submission"] - test_wav_names
overlap = pd.Series(
    {
        "Train ∩ test basenames": len(names["train"] & names["test"]),
        "Train ∩ submission basenames": len(
            names["train"] & names["sample_submission"]
        ),
        "Test ∩ submission basenames": len(names["test"] & names["sample_submission"]),
        "Submission basenames absent from test WAVs": len(missing_submission_audio),
    },
    name="count",
)
display(overlap.to_frame())
print("Submission matches test filenames:", names["sample_submission"] == names["test"])

## 5. Target distribution
Analyze training labels only. Quantized scores and unequal frequencies require
care in future validation, especially rare 1.0 and 1.5 categories. Do not assume
normality or implement a validation strategy here. Label 0 remains unchanged:
its meaning requires investigation because the rubric primarily describes 1–5.

In [ ]:
labels = train["label"]
label_counts = labels.value_counts().sort_index().rename("count")
display(labels.describe().to_frame("training label"))
print("Unique labels:", labels.nunique())
print("Observed target range:", labels.min(), "to", labels.max())
display(label_counts.to_frame())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(labels, bins=np.arange(-0.25, 5.76, 0.5), edgecolor="white")
axes[0].set(
    title="Training target histogram", xlabel="Grammar score", ylabel="Recordings"
)
axes[1].bar(label_counts.index, label_counts.values, width=0.35)
axes[1].set(
    title="Exact label frequencies", xlabel="Grammar score", ylabel="Recordings"
)
fig.tight_layout()
plt.show()

## 6. Audio metadata
The metadata tables were constructed with `inspect_dataset_audio`, which uses
the production WAV metadata collector. Each row has a split and full path;
`(split, filename)` is the identity used here. Unreferenced WAVs, if present,
remain visible in metadata inspection.

In [ ]:
for split, frame in metadata.items():
    print(split)
    display(frame.head())
    print("Unreferenced readable WAVs:", len(set(frame["filename"]) - names[split]))

## 7. Audio duration analysis
Descriptive statistics use seconds. Pandas `describe` reports sample standard
deviation; population standard deviation is also shown to match the inspection
report convention. Shared histogram bins and an ECDF allow direct comparison.
Visual differences alone do not establish statistical or causal significance.

In [ ]:
duration_stats = pd.DataFrame(
    {split: frame["duration_seconds"].describe() for split, frame in metadata.items()}
)
duration_stats.loc["population std"] = [
    metadata[split]["duration_seconds"].std(ddof=0) for split in duration_stats.columns
]
display(duration_stats)
duration_bins = np.linspace(0, audio_metadata["duration_seconds"].max() + 1, 22)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for ax, (split, frame) in zip(axes, metadata.items(), strict=True):
    ax.hist(frame["duration_seconds"], bins=duration_bins, edgecolor="white")
    ax.set(
        title=f"{split.title()} duration histogram",
        xlabel="Duration (seconds)",
        ylabel="Recordings",
    )
fig.tight_layout()
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for split, frame in metadata.items():
    values = frame["duration_seconds"].to_numpy()
    axes[0].hist(values, bins=duration_bins, density=True, alpha=0.5, label=split)
    ordered = np.sort(values)
    axes[1].step(
        ordered,
        np.arange(1, len(ordered) + 1) / len(ordered),
        where="post",
        label=split,
    )
axes[0].set(
    title="Train/test duration density",
    xlabel="Duration (seconds)",
    ylabel="Density (per second)",
)
axes[1].set(
    title="Train/test duration ECDF",
    xlabel="Duration (seconds)",
    ylabel="Cumulative proportion",
)
for ax in axes:
    ax.legend()
fig.tight_layout()
plt.show()
print(
    "Train minus test mean duration (seconds):",
    duration_stats.loc["mean", "train"] - duration_stats.loc["mean", "test"],
)

## 8. Duration vs grammar score
Join labels only to training metadata. A one-to-one merge and explicit coverage
checks prevent duplicated or missing training rows. Correlations use original
values. Scatter transparency handles overlap without altering data. P-values
are descriptive and do not make an exploratory association causal.

In [ ]:
assert train_metadata["split"].eq("train").all()
train_with_duration = train.merge(
    train_metadata, on="filename", how="left", validate="one_to_one", indicator=True
)
assert len(train_with_duration) == len(train)
assert train_with_duration["_merge"].eq("both").all()
assert train_with_duration["duration_seconds"].notna().all()
print(f"Verified one-to-one train merge: {len(train_with_duration)} matched rows")
correlations = {}
for name, function in [("Pearson", pearsonr), ("Spearman", spearmanr)]:
    result = function(
        train_with_duration["duration_seconds"], train_with_duration["label"]
    )
    correlations[name] = {"coefficient": result.statistic, "p_value": result.pvalue}
display(pd.DataFrame(correlations).T)
fig, ax = plt.subplots()
ax.scatter(
    train_with_duration["duration_seconds"],
    train_with_duration["label"],
    alpha=0.25,
    s=18,
)
ax.set(
    title="Training duration vs grammar score",
    xlabel="Duration (seconds)",
    ylabel="Grammar score",
)
plt.show()
display(
    train_with_duration.groupby("label")["duration_seconds"].agg(
        count="count",
        mean_duration="mean",
        median_duration="median",
        std_duration="std",
    )
)

## 9. Short recording analysis
For exploration, short means strictly less than 30 seconds. The training subset
is small, so conclusions should not be drawn from it alone. Test labels are
excluded from the table. The shortest test recording is explicitly identified.

In [ ]:
short_train = train_with_duration.loc[
    train_with_duration["duration_seconds"] < SHORT_SECONDS,
    ["filename", "duration_seconds", "label"],
].sort_values(["duration_seconds", "filename"])
short_test = test_metadata.loc[
    test_metadata["duration_seconds"] < SHORT_SECONDS,
    ["filename", "duration_seconds"],
].sort_values(["duration_seconds", "filename"])
display(short_train)
display(short_test)
display(
    pd.DataFrame(
        {
            "overall train": labels.agg(["count", "mean", "median"]),
            "short train": short_train["label"].agg(["count", "mean", "median"]),
        }
    )
)
print("Shortest test recording:")
display(
    test_metadata.nsmallest(1, "duration_seconds")[["filename", "duration_seconds"]]
)

## 10. Label-0 investigation
Keep all label-0 samples. Duration comparisons do not reveal what zero means
semantically. Investigate the competition rubric and data documentation before
making any decision about their treatment.

In [ ]:
zero_duration = train_with_duration.loc[
    train_with_duration["label"].eq(0), "duration_seconds"
]
positive_duration = train_with_duration.loc[
    train_with_duration["label"].gt(0), "duration_seconds"
]
display(
    zero_duration.agg(["count", "mean", "median", "min", "max"]).to_frame(
        "label-0 duration (seconds)"
    )
)
fig, ax = plt.subplots()
for name, values in [("label == 0", zero_duration), ("label > 0", positive_duration)]:
    ax.hist(values, bins=duration_bins, density=True, alpha=0.5, label=name)
ax.set(
    title="Training duration by zero/positive label",
    xlabel="Duration (seconds)",
    ylabel="Density (per second)",
)
ax.legend()
plt.show()

## 11. Train/test distribution comparison
Compare audio format in a table rather than plotting constants. Standardized
header formats do not imply identical duration distributions.

In [ ]:
comparison = pd.DataFrame(
    {
        split: {
            "Mean duration (seconds)": frame["duration_seconds"].mean(),
            "Median duration (seconds)": frame["duration_seconds"].median(),
            "Sample rates (Hz)": sorted(frame["sample_rate"].unique().tolist()),
            "Channels": sorted(frame["num_channels"].unique().tolist()),
            "Sample widths (bytes)": sorted(
                frame["sample_width_bytes"].unique().tolist()
            ),
        }
        for split, frame in metadata.items()
    }
)
display(comparison)

## 12. Key findings
The Markdown summary below is generated from computed results, with no
hard-coded derived statistics.

In [ ]:
from IPython.display import Markdown, display

rare_counts = label_counts.reindex([1.0, 1.5], fill_value=0)
standard_format = all(
    audio_metadata[column].nunique() == 1
    for column in ["sample_rate", "num_channels", "sample_width_bytes"]
)
summary = f"""
- Dataset sizes: **{len(train)} train**, **{len(test)} test**, and
  **{len(submission)} sample-submission rows**.
- Training targets range from **{labels.min():g} to {labels.max():g}**, with
  **{labels.nunique()} discrete values**: {label_counts.index.tolist()}.
  Frequencies range from **{label_counts.min()} to {label_counts.max()}**.
- Rare categories: **1.0 has {rare_counts.loc[1.0]}** samples and
  **1.5 has {rare_counts.loc[1.5]}**. Label 0 has **{len(zero_duration)}** samples;
  its semantics remain unresolved and all samples are retained.
- All inspected recordings use a consistent audio format:
  Rates: {sorted(audio_metadata["sample_rate"].unique().tolist())} Hz;
  channels: {sorted(audio_metadata["num_channels"].unique().tolist())};
  sample width: {sorted(audio_metadata["sample_width_bytes"].unique().tolist())} bytes.
- Duration means: **{duration_stats.loc["mean", "train"]:.3f} s train** and
  **{duration_stats.loc["mean", "test"]:.3f} s test**; medians:
  **{duration_stats.loc["50%", "train"]:.3f} s** and
  **{duration_stats.loc["50%", "test"]:.3f} s**, respectively.
- Recordings below {SHORT_SECONDS} s: **{len(short_train)} train** and
  **{len(short_test)} test**. The small subset supports descriptive inspection only.
- Submission and test basename sets match:
  **{names["sample_submission"] == names["test"]}**;
  **{len(missing_submission_audio)}** submission basenames are absent from downloaded
  test WAVs. This packaging/submission issue requires investigation before submission.
  Cross-split basename overlap does not establish sample identity.
"""
display(Markdown(summary))

## 13. Modeling implications / next steps
Future work should investigate robust cross-validation and rare target strata,
label-0 semantics, duration as a potential feature, possible train/test duration
shift, and ASR/transcript-based grammar modeling. Resolve the sample-submission
packaging mismatch before final submission. These are tentative directions;
this notebook implements no CV, models, transcription, feature engineering,
audio signal processing, or submission generation.